# VAD Detection with Silero VAD

Run Silero VAD on WAV files and save raw per-frame speech probabilities.
Each frame is 32ms (512 samples at 16kHz). Save raw probs for visualization
so we can pick threshold/filtering parameters later.

In [ ]:
from pathlib import Path
import numpy as np

WAV_DIR = Path("../data/wav")
OUTPUT_DIR = Path("../data/vad_probs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000
WINDOW_SIZE = 512  # 32ms per frame at 16kHz

In [ ]:
import torch

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True,
)

(get_speech_timestamps, _, read_audio, _, _) = utils

# Move to MPS if available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print(f"Silero VAD loaded on {device}")

In [ ]:
wav_files = sorted(WAV_DIR.glob("*.wav"))
print(f"Found {len(wav_files)} WAV files in {WAV_DIR}")

In [ ]:
from tqdm.auto import tqdm

for wav_path in tqdm(wav_files, desc="Files"):
    wav = read_audio(str(wav_path), sampling_rate=SAMPLE_RATE).to(device)

    # Extract per-frame speech probabilities
    chunks = range(0, len(wav), WINDOW_SIZE)
    probs = []
    for start in tqdm(chunks, desc=wav_path.stem, leave=False):
        chunk = wav[start : start + WINDOW_SIZE]
        if len(chunk) < WINDOW_SIZE:
            chunk = torch.nn.functional.pad(chunk, (0, WINDOW_SIZE - len(chunk)))
        prob = model(chunk, SAMPLE_RATE).item()
        probs.append(prob)
    model.reset_states()

    # Save as .npy — one float32 per 32ms frame
    out_path = OUTPUT_DIR / f"{wav_path.stem}.npy"
    np.save(out_path, np.array(probs, dtype=np.float32))

print(f"\nDone. Saved {len(wav_files)} prob arrays to {OUTPUT_DIR}")

In [ ]:
# Preview: plot speech probabilities
import matplotlib.pyplot as plt

npy_files = sorted(OUTPUT_DIR.glob("*.npy"))
probs = np.load(npy_files[0])
time_s = np.arange(len(probs)) * WINDOW_SIZE / SAMPLE_RATE

# Overview (full file, downsampled for readability)
fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=False)

# 1) Full file overview — use fill to show density
axes[0].fill_between(time_s, probs, alpha=0.6)
axes[0].set_title(f"{npy_files[0].stem} — full ({time_s[-1]:.0f}s)")
axes[0].set_ylim(0, 1)
axes[0].axhline(y=0.5, color="r", linestyle="--", alpha=0.5, label="0.5")
axes[0].axhline(y=0.1, color="orange", linestyle="--", alpha=0.5, label="0.1")
axes[0].legend(loc="upper right")

# 2) First 60s zoomed in
mask = time_s <= 60
axes[1].fill_between(time_s[mask], probs[mask], alpha=0.6)
axes[1].plot(time_s[mask], probs[mask], linewidth=0.3, color="C0")
axes[1].set_title("First 60s")
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.5, color="r", linestyle="--", alpha=0.5)
axes[1].axhline(y=0.1, color="orange", linestyle="--", alpha=0.5)

# 3) Histogram of probabilities
axes[2].hist(probs, bins=100, edgecolor="none", alpha=0.7)
axes[2].set_xlabel("Speech probability")
axes[2].set_ylabel("Frame count")
axes[2].set_title("Probability distribution")
axes[2].axvline(x=0.5, color="r", linestyle="--", alpha=0.5, label="0.5")
axes[2].axvline(x=0.1, color="orange", linestyle="--", alpha=0.5, label="0.1")
axes[2].legend()

for ax in axes[:2]:
    ax.set_ylabel("P(speech)")

plt.tight_layout()
plt.show()